# Feature Scaling — Standardization

Many ML algorithms are sensitive to the **scale** of features — if `Age` ranges 18–60 and `EstimatedSalary` ranges 15,000–150,000, a model can end up implicitly treating salary as "more important" purely because its numbers are bigger, not because it actually matters more. This notebook covers **standardization** (`StandardScaler`): rescaling every feature to have mean 0 and standard deviation 1.

## 1. Load and Prepare the Data

`Social_Network_Ads.csv` — dropping the first two columns (`User ID`, `Gender`) keeps only the numeric features (`Age`, `EstimatedSalary`) and the target (`Purchased`).

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('Social_Network_Ads.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'Social_Network_Ads.csv'

In [ ]:
df=df.iloc[:,2:]

In [ ]:
df.sample(5)

## Train test split

`train_test_split()` sets aside 30% of the data for testing. This has to happen **before** scaling — the scaler is only allowed to learn from the training set, never the test set (more on why below).

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('Purchased', axis=1),
                                                    df['Purchased'],
                                                    test_size=0.3,
                                                    random_state=0)

X_train.shape, X_test.shape

# StandardScaler

`StandardScaler` transforms each feature to **mean 0, standard deviation 1**, using the formula `(x - mean) / std`. Two separate steps here:
- `.fit(X_train)` — **learns** the mean and standard deviation of each column, from the training data only
- `.transform(...)` — **applies** that learned formula to rescale the data

### Why Fit Only on `X_train`?
This notebook already does this correctly — worth calling out why it matters: if the scaler were fit on the full dataset (train + test combined), information about the test set's distribution would leak into training via the mean/std used — a subtle form of **data leakage** that makes test performance look better than it would in the real world, where future data is genuinely unseen.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit the scaler to the train set, it will learn the parameters
scaler.fit(X_train)

# transform train and test sets
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

### A Shortcut: `fit_transform()`

Calling `.fit()` then `.transform()` on the *same* data (as done for `X_train` above) is common enough that scikit-learn provides a single combined method. Functionally identical to the two-step version — this is just the more common way you'll see it written in practice.

In [ ]:
X_train_scaled_shortcut = scaler.fit_transform(X_train)
# equivalent to: scaler.fit(X_train); X_train_scaled = scaler.transform(X_train)


### Inspecting What the Scaler Learned

`scaler.mean_` shows the mean it learned for each column (from `X_train` only). `scaler.scale_` (the standard deviation) is the other half of the formula, worth checking alongside it.

In [ ]:
scaler.mean_

In [ ]:
scaler.scale_


In [ ]:
X_train

In [ ]:
X_train_scaled

`.transform()` returns a plain **NumPy array**, stripping away the column names and index that a DataFrame has — wrapping it back into a DataFrame (next cell) makes it easier to keep working with named columns.

In [ ]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

### Reversing a Scale — `inverse_transform()`

Sometimes you need to convert scaled values back to their original units — e.g. to display a prediction in real dollars/years rather than standardized units. `inverse_transform()` undoes exactly what `transform()` did.

In [ ]:
scaler.inverse_transform(X_train_scaled)[:5]   # back to original Age/Salary values


### Confirming the Effect Numerically

After scaling, every column's mean should be ~0 and standard deviation ~1 — compare against the original stats.

In [ ]:
np.round(X_train.describe(), 1)

In [ ]:
np.round(X_train_scaled.describe(), 1)

## 2. Visualizing the Effect of Scaling

# Effect of Scaling

The scatter plot's *shape* (the relationship between `Age` and `EstimatedSalary`) doesn't change — scaling doesn't distort relationships between points, it only changes the **axis scale** they're measured on (compare the axis numbers between the two plots).

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.scatter(X_train['Age'], X_train['EstimatedSalary'])
ax1.set_title("Before Scaling")
ax2.scatter(X_train_scaled['Age'], X_train_scaled['EstimatedSalary'],color='red')
ax2.set_title("After Scaling")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# before scaling
ax1.set_title('Before Scaling')
sns.kdeplot(X_train['Age'], ax=ax1)
sns.kdeplot(X_train['EstimatedSalary'], ax=ax1)

# after scaling
ax2.set_title('After Standard Scaling')
sns.kdeplot(X_train_scaled['Age'], ax=ax2)
sns.kdeplot(X_train_scaled['EstimatedSalary'], ax=ax2)
plt.show()

Similarly, each column's **distribution shape** is preserved — scaling shifts and stretches/compresses the x-axis, but doesn't change the underlying pattern.

# Comparison of Distributions

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# before scaling
ax1.set_title('Age Distribution Before Scaling')
sns.kdeplot(X_train['Age'], ax=ax1)

# after scaling
ax2.set_title('Age Distribution After Standard Scaling')
sns.kdeplot(X_train_scaled['Age'], ax=ax2)
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# before scaling
ax1.set_title('Salary Distribution Before Scaling')
sns.kdeplot(X_train['EstimatedSalary'], ax=ax1)

# after scaling
ax2.set_title('Salary Distribution Standard Scaling')
sns.kdeplot(X_train_scaled['EstimatedSalary'], ax=ax2)
plt.show()

# Why scaling is important?

The real-world test of whether scaling matters: train the same two model types on scaled vs. unscaled data, and compare accuracy.

### Logistic Regression — Distance/Gradient-Based, Scale-Sensitive

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression()
lr_scaled = LogisticRegression()

In [ ]:
lr.fit(X_train,y_train)
lr_scaled.fit(X_train_scaled,y_train)

In [ ]:
y_pred = lr.predict(X_test)
y_pred_scaled = lr_scaled.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
print("Actual",accuracy_score(y_test,y_pred))
print("Scaled",accuracy_score(y_test,y_pred_scaled))

Logistic regression's optimizer (gradient descent) converges faster and more reliably when features are on similar scales — with very different-scaled features, it can take a very different (sometimes worse) path to a solution.

### Decision Tree — Scale-Invariant

By contrast, tree-based models split on "is this feature above or below some threshold?" — a **rank-based** decision, not a distance calculation. Scaling a feature doesn't change which rows fall above/below a given threshold, so trees (and forests, and gradient-boosted trees) typically show little to no accuracy difference before vs. after scaling — you should see that reflected in the numbers below.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dt = DecisionTreeClassifier()
dt_scaled = DecisionTreeClassifier()

In [ ]:
dt.fit(X_train,y_train)
dt_scaled.fit(X_train_scaled,y_train)

In [ ]:
y_pred = dt.predict(X_test)
y_pred_scaled = dt_scaled.predict(X_test_scaled)

In [ ]:
print("Actual",accuracy_score(y_test,y_pred))
print("Scaled",accuracy_score(y_test,y_pred_scaled))

In [ ]:
df.describe()

# Effect of Outlier

Adding a few extreme outliers (`Age` of 90–95, `EstimatedSalary` of 250,000–350,000) tests how sensitive `StandardScaler` is to them — since it uses the **mean** and **standard deviation**, both of which outliers can distort significantly, unlike the median.

### ⚠️ Important: `df.append()` Is Deprecated (and Removed in Pandas 2.0+)

Same issue as in the earlier scraping notebooks — `DataFrame.append()` no longer exists on current pandas. Use `pd.concat()` instead.

In [ ]:
df = df.append(pd.DataFrame({'Age':[5,90,95],'EstimatedSalary':[1000,250000,350000],'Purchased':[0,1,1]}),ignore_index=True)

In [ ]:
df = pd.concat([
    df,
    pd.DataFrame({'Age': [5, 90, 95], 'EstimatedSalary': [1000, 250000, 350000], 'Purchased': [0, 1, 1]})
], ignore_index=True)


In [ ]:
df

In [ ]:
plt.scatter(df['Age'], df['EstimatedSalary'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('Purchased', axis=1),
                                                    df['Purchased'],
                                                    test_size=0.3,
                                                    random_state=0)

X_train.shape, X_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit the scaler to the train set, it will learn the parameters
scaler.fit(X_train)

# transform train and test sets
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.scatter(X_train['Age'], X_train['EstimatedSalary'])
ax1.set_title("Before Scaling")
ax2.scatter(X_train_scaled['Age'], X_train_scaled['EstimatedSalary'],color='red')
ax2.set_title("After Scaling")
plt.show()

Notice how the scaled scatter plot's axis range has stretched dramatically compared to the earlier one — a handful of extreme values pulled the mean and standard deviation far enough that most of the *normal* data now gets squeezed into a small region near the center.

## 3. The Fix for Outlier Sensitivity — `RobustScaler`

`RobustScaler` uses the **median** and **interquartile range (IQR)** instead of mean/standard deviation — both are far less affected by a handful of extreme values. Same `fit`/`transform` interface as `StandardScaler`, just a more outlier-resistant choice when a dataset has (or might have) them.

In [ ]:
from sklearn.preprocessing import RobustScaler

robust_scaler = RobustScaler()
X_train_robust = robust_scaler.fit_transform(X_train)
X_test_robust = robust_scaler.transform(X_test)

X_train_robust = pd.DataFrame(X_train_robust, columns=X_train.columns)

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))
ax1.scatter(X_train_scaled['Age'], X_train_scaled['EstimatedSalary'])
ax1.set_title('StandardScaler (with outliers)')
ax2.scatter(X_train_robust['Age'], X_train_robust['EstimatedSalary'], color='green')
ax2.set_title('RobustScaler (with outliers)')
plt.show()


## Summary

| Task | Method |
|---|---|
| Split before scaling | `train_test_split(X, y, test_size=...)` |
| Learn scaling params from train only | `scaler.fit(X_train)` |
| Apply the scaling | `scaler.transform(X)` |
| Fit + transform in one step | `scaler.fit_transform(X_train)` |
| Inspect learned parameters | `scaler.mean_`, `scaler.scale_` |
| Undo scaling | `scaler.inverse_transform(X_scaled)` |
| Combine train+test rows safely | list + `pd.concat(...)` *(not the removed `df.append()`)* |
| Outlier-resistant scaling | `RobustScaler()` — same interface, uses median/IQR instead of mean/std |

### Which models actually need scaling?
- **Sensitive to scale:** logistic regression, SVMs, KNN, neural networks, PCA — anything based on distances or gradient descent.
- **Scale-invariant:** decision trees, random forests, gradient-boosted trees — anything based on threshold splits.